# Phase C: Build Dataset + Train + Evaluate (Steps 08-10)

All-in-one Kaggle notebook on P100 GPU.

1. Downloads stems + curves from HF Hub `Uday-4/djmix-v3`
2. Step 08: Builds training dataset (mel spectrograms + train/val/test splits)
3. Step 09: Trains StemTransitionNet (CNN+Transformer)
4. Step 10: Evaluates on test set
5. Uploads best model checkpoint to HF Hub

**Runtime:** P100 GPU (Settings > Accelerator > GPU P100).

**Estimated time:** ~1.5-2.5 hours total.

In [ ]:
# Cell 1 — Install dependencies + clone repo
!pip install -q librosa soundfile huggingface_hub torch torchaudio cvxpy ecos clarabel numpy scipy

import subprocess, sys

# Clone the v4 repo for code + manifest
# If the repo is private, use: https://{GITHUB_PAT}@github.com/Uday-461/ai-dj-v4.git
GITHUB_PAT = ""  # <-- paste GitHub PAT here if repo is private (leave empty if public)
clone_url = f"https://{GITHUB_PAT}@github.com/Uday-461/ai-dj-v4.git" if GITHUB_PAT else "https://github.com/Uday-461/ai-dj-v4.git"
result = subprocess.run(
    ['git', 'clone', '--depth', '1', clone_url, '/kaggle/working/ai-dj-v4'],
    capture_output=True, text=True
)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print('Clone failed:', result.stderr)
else:
    print('Repo ready at /kaggle/working/ai-dj-v4')

# Install the aidj package
!pip install -q -e /kaggle/working/ai-dj-v4
sys.path.insert(0, '/kaggle/working/ai-dj-v4')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2 — Config + selective download from HF Hub
import os
import time
from pathlib import Path
from huggingface_hub import snapshot_download

# ---- EDIT THESE ----
HF_TOKEN   = "hf_..."                          # <-- paste your HuggingFace token
HF_REPO    = "Uday-4/djmix-v3"                 # HF dataset repo
MODEL_REPO = "Uday-4/aidj-v3-stemtransitionnet" # HF model repo for checkpoint upload
# ---------------------

DATA_ROOT  = "/kaggle/working/djmix"
REPO_ROOT  = "/kaggle/working/ai-dj-v4"

os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["AIDJ_DATA_ROOT"] = DATA_ROOT

if HF_TOKEN.startswith("hf_..."):
    raise ValueError("Edit Cell 2: set HF_TOKEN to your real HuggingFace token")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    api.whoami()
    print(f'HF auth OK')
except Exception as e:
    raise RuntimeError(f'HF auth failed: {e}')

# Selective download: skip mix audio + track MP3s (only need stems, curves, transitions, residuals)
print(f'\nDownloading data from HF Hub ({HF_REPO})...')
t0 = time.time()
snapshot_download(
    repo_id=HF_REPO,
    repo_type="dataset",
    local_dir=DATA_ROOT,
    token=HF_TOKEN,
    allow_patterns=[
        "results/transitions/*.pkl",       # transition metadata (~1MB)
        "results/stem_curves/**/*.npz",    # ground truth curves (~10MB)
        "results/residuals/**/*.npz",      # residual spectrograms (~50MB)
        "stems/tracks/**/*.ogg",           # track stems for spectrograms (~5GB)
    ],
)
elapsed = time.time() - t0
print(f'Download complete in {elapsed/60:.1f} min')

# Verify what we got
data_root = Path(DATA_ROOT)
n_pkl  = len(list((data_root / "results" / "transitions").glob("*.pkl"))) if (data_root / "results" / "transitions").exists() else 0
n_npz  = len(list((data_root / "results" / "stem_curves").rglob("*.npz"))) if (data_root / "results" / "stem_curves").exists() else 0
n_res  = len(list((data_root / "results" / "residuals").rglob("*.npz"))) if (data_root / "results" / "residuals").exists() else 0
n_ogg  = len(list((data_root / "stems" / "tracks").rglob("*.ogg"))) if (data_root / "stems" / "tracks").exists() else 0
print(f'\nDownloaded: {n_pkl} transition PKLs, {n_npz} curve NPZs, {n_res} residual NPZs, {n_ogg} stem OGGs')

# Manifest path: from the cloned git repo
MANIFEST = os.path.join(REPO_ROOT, "data", "manifest_50mix.json")
if not os.path.exists(MANIFEST):
    raise FileNotFoundError(f"Manifest not found at {MANIFEST}")
print(f'Manifest: {MANIFEST}')

In [ ]:
# Cell 3 — Step 08: Build dataset (compute spectrograms + train/val/test split)
import subprocess, sys, time

TRAINING_DIR = os.path.join(DATA_ROOT, "training")

print("=" * 60)
print("Step 08: Build Dataset")
print("=" * 60)

t0 = time.time()
result = subprocess.run([
    sys.executable, os.path.join(REPO_ROOT, "scripts", "08_build_dataset.py"),
    "--manifest", MANIFEST,
    "--data-root", DATA_ROOT,
    "--output-dir", TRAINING_DIR,
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError(f"Step 08 failed with exit code {result.returncode}")

elapsed = time.time() - t0
print(f'\nStep 08 complete in {elapsed/60:.1f} min')

# Verify splits
import json
for split in ["train", "val", "test"]:
    manifest_path = os.path.join(TRAINING_DIR, split, "manifest.json")
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            m = json.load(f)
        print(f'  {split}: {m["num_samples"]} samples')

        # Verify first sample's input.npz shape
        if m["samples"]:
            import numpy as np
            curves_path = m["samples"][0]["curves_path"]
            input_path = curves_path.replace(".npz", ".input.npz")
            if os.path.exists(input_path):
                data = np.load(input_path)
                print(f'    input.npz shape: {data["spectrograms"].shape}')
    else:
        print(f'  {split}: MISSING')

In [ ]:
# Cell 4 — Step 09: Train StemTransitionNet + Step 10: Evaluate
import subprocess, sys, time

CHECKPOINT_DIR = os.path.join(DATA_ROOT, "checkpoints")
EPOCHS     = 50
BATCH_SIZE = 16
LR         = 1e-4
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# --- Step 09: Train ---
print("=" * 60)
print(f"Step 09: Train ({EPOCHS} epochs, batch_size={BATCH_SIZE}, device={DEVICE})")
print("=" * 60)

t0 = time.time()
result = subprocess.run([
    sys.executable, os.path.join(REPO_ROOT, "scripts", "09_train.py"),
    "--train-dir", os.path.join(TRAINING_DIR, "train"),
    "--val-dir", os.path.join(TRAINING_DIR, "val"),
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--device", DEVICE,
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError(f"Step 09 failed with exit code {result.returncode}")

elapsed = time.time() - t0
print(f'\nStep 09 complete in {elapsed/60:.1f} min')

# --- Step 10: Evaluate ---
best_model = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if not os.path.exists(best_model):
    raise FileNotFoundError(f"No best_model.pt found at {best_model}")

print(f'\n{"=" * 60}')
print("Step 10: Evaluate on test set")
print("=" * 60)

result = subprocess.run([
    sys.executable, os.path.join(REPO_ROOT, "scripts", "10_evaluate.py"),
    "--model", best_model,
    "--test-dir", os.path.join(TRAINING_DIR, "test"),
    "--device", DEVICE,
], capture_output=False)

if result.returncode != 0:
    print(f"Step 10 failed with exit code {result.returncode}")

# Show checkpoint file size
import os
size_mb = os.path.getsize(best_model) / (1024 * 1024)
print(f'\nCheckpoint: {best_model} ({size_mb:.1f} MB)')

In [ ]:
# Cell 5 — Upload best model to HF Hub
from huggingface_hub import HfApi

best_model = os.path.join(DATA_ROOT, "checkpoints", "best_model.pt")
if not os.path.exists(best_model):
    raise FileNotFoundError(f"No checkpoint found at {best_model}")

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=MODEL_REPO, repo_type="model", private=True, exist_ok=True)

print(f'Uploading {best_model} -> {MODEL_REPO}')
api.upload_file(
    path_or_fileobj=best_model,
    path_in_repo="best_model.pt",
    repo_id=MODEL_REPO,
    repo_type="model",
    commit_message="Upload trained StemTransitionNet checkpoint",
)
print(f'Done! Model at: https://huggingface.co/{MODEL_REPO}')